# 🫀 퀘스트 46 · Q1 — **`GMIN_S` 를 지표 해상도로 다시 정의한다**

| | **MedKOS / `notebooks/quest46_q1_gmin_resolution.ipynb`** |
|---|---|
| 퀘스트 | `ailab-2026-0046` — 맥락정의 클래스(S)의 평가·보정 프로토콜 |
| 출발점 | `ailab-2026-0045` (실험22-A) |
| 규약 | `pipelines/SCORING_RULES.md` **R7 · R8 · R11 · R11-b** |

## 왜 이 실험인가

실험22-A 에서 **S 관련 결론이 전부 미결**로 나왔다. 효과가 없어서가 아니라
**채점 환자가 within 8명 / cross 11명**뿐이라 환자 부트스트랩 CI 가 시드 CI 의
**3.3배**로 벌어졌기 때문이다.

그리고 `GMIN_S=20` 은 잘못 고른 값이었다 — S 가 28개인 #213 은 R-prec 해상도가
`1/28 = 3.6%p` 라 Δ 가 **+0.4857** 까지 튀었고, 그 하나가 매크로의 **110%** 를
만들었다(→ R11-b).

**이 실험은 모델을 안 건드린다. 채점 대상 선정 규칙만 바꾼다.** 학습 0회.

## 무엇을 바꾸나

`GMIN` 을 **계단 크기**로 정하면 지표마다 답이 달라진다. AUROC 는 계단이
`1/(n_pos·n_neg)` 로 촘촘해서 `n_pos=2` 도 통과한다. 그래서 기준을
**환자별 지표의 표집 SE**(Hanley–McNeil)로 바꾼다 — 지표에 덜 의존하고,
"이 환자 한 명의 값을 얼마나 믿을 수 있나" 를 직접 잰다.

## 사전등록

| 관문 | 내용 |
|---|---|
| **Q1-1** | INCART S 채점 환자 **≥ 20명** (SE 조건 하에서) |
| **Q1-2** | 환자별 AUROC 중앙 SE ≤ **0.05** · 최대 SE ≤ 0.10 |
| **Q1-3** | 채점 환자가 덮는 S ≥ **70%** |
| **Q1-4** | 채점 환자 안 지배 지분 ≤ **50%** (R11-3) |
| **Q1-5** | INCART 가 MIT-BIH DS2 보다 채점 환자를 더 많이 준다 |

⚠️ **한계**: INCART 의 `pid` 는 레코드(75)이고 실제 환자는 32명이다(§6.5 L3).
여기서 '환자' 는 레코드를 뜻하며 환자 단위 CI 는 그만큼 **낙관적**이다.


In [ ]:
# CELL 0 — 공용 사전점검 (pipelines/SCORING_RULES.md)
import numpy as np
from scipy import stats

def decide(lo, hi, thr, direction):
    """사전등록 관문의 유일한 계약: 지지 / 기각 / **미결**."""
    if direction not in (">", "<"):
        raise ValueError("direction 은 '>' 또는 '<'")
    if direction == ">":
        if lo > thr: return "✅ 지지"
        if hi < thr: return "❌ 기각"
    else:
        if hi < thr: return "✅ 지지"
        if lo > thr: return "❌ 기각"
    return "⚠️ 미결"

def t_ci(v, conf=0.95):
    v = np.asarray([x for x in v if np.isfinite(x)], float); n = len(v)
    m = float(v.mean()) if n else float("nan")
    if n < 2: return m, np.nan, np.nan
    h = float(stats.t.ppf(.5 + conf / 2, n - 1) * v.std(ddof=1) / np.sqrt(n))
    return m, m - h, m + h

class AssetError(RuntimeError): pass
print("CELL 0 ✅ decide · t_ci 준비")

In [ ]:
# CELL 1 — 설정
import os, sys, json, time, subprocess, importlib
importlib.invalidate_caches()
try:
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
except Exception as e:
    print("⚠️ Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
MITBIH  = os.path.join(DRIVE_ROOT, "mitbih")
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun

# ★ 학습 0회. 실험22-A 가 남긴 예측 캐시만 읽는다.
CACHE = os.path.join(PROJECT, "data", "exp22a_probs_s5_v1.npz")

SE_CAP   = 0.05     # 환자별 AUROC 의 표집 SE 상한(중앙값)
SE_MULT  = 2.0      # 최대 SE 는 SE_CAP × 이 배수 이내 — #213 같은 시한폭탄 차단
N_MIN    = 20       # 채점 환자 하한 (Q1 합격 기준)
COV_MIN  = 0.70     # 채점 환자가 덮어야 할 양성 비율
DOM_MAX  = 0.50     # 채점 환자 안에서의 지배 지분 상한 (R11-3)
GMINS    = [2, 5, 10, 15, 20, 30, 40, 50, 75, 100, 150, 200]
SEED0    = 20260802

_DS1 = [101,106,108,109,112,114,115,116,118,119,122,124,201,203,205,207,208,209,215,220,223,230]
_DS2 = [100,103,105,111,113,117,121,123,200,202,210,212,213,214,219,221,222,228,231,232,233,234]

CONFIG = dict(
    exp="quest46_q1_gmin_resolution", quest="ailab-2026-0046", step="gmin-resolution",
    parent_exp=["exp22a_axis_transfer", "ailab-2026-0045"],
    purpose=("실험22-A 에서 S 결론이 **전부 미결**로 나온 이유는 채점 환자가 8/11명뿐이라 "
             "환자 부트스트랩 CI 가 시드 CI 의 3.3배로 벌어졌기 때문이다. "
             "`GMIN_S` 를 **지표 해상도**로 다시 정의해 채점 환자를 몇 명까지 확보할 수 "
             "있는지 실측한다. 학습 0회 — 캐시된 확률만 쓴다"),
    dataset="MIT-BIH DS2(within · 대조군) + INCART(cross · 본 대상)",
    change_one_thing="모델·확률 그대로. **채점 대상 선정 규칙만** 바꾼다",
    thresholds=dict(se_cap=SE_CAP, se_mult=SE_MULT, n_min=N_MIN,
                    cov_min=COV_MIN, dom_max=DOM_MAX),
    why_se=("`GMIN` 을 **계단 크기**(R-prec 의 1/n_pos)로 정하면 지표마다 답이 달라진다. "
            "AUROC 는 계단이 1/(n_pos·n_neg) 로 촘촘해 n_pos=2 도 통과한다. "
            "그래서 기준을 **표집 SE** 로 잡는다 — 두 지표 모두에서 '이 환자 한 명의 값을 "
            "얼마나 믿을 수 있나' 를 잰다"),
    predictions={
        "Q1-1": f"INCART 채점 환자 ≥ {N_MIN}명 (SE 조건 하에서)",
        "Q1-2": f"선택된 GMIN 에서 환자별 AUROC 중앙 SE ≤ {SE_CAP}",
        "Q1-3": f"채점 환자가 덮는 S ≥ {COV_MIN:.0%}",
        "Q1-4": f"채점 환자 안 지배 지분 ≤ {DOM_MAX:.0%}",
        "Q1-5": "INCART 가 MIT-BIH DS2 보다 채점 환자를 더 많이 준다"},
    caveat=("INCART 의 `pid` 는 레코드(75)이고 실제 환자는 32명이다(§6.5 L3). "
            "여기서 '환자' 는 **레코드**를 뜻한다 — 환자 단위 CI 는 그만큼 낙관적이다. "
            "Hanley SE 는 보수적 근사라(부트 대비 1.1~1.6배) 하한 설정용으로만 쓴다"))
np.random.seed(SEED0)
run = MedKOSRun("quest46_q1_gmin", CONFIG, project=PROJECT)
run.log(f"설정 ✅ SE_CAP {SE_CAP} · N_MIN {N_MIN} · 캐시 {os.path.basename(CACHE)}")

In [ ]:
# CELL 2 — 【G0】 자산·정합성 점검. 가정이 틀리면 여기서 멈춘다
need = {CACHE: "실험22-A 예측 캐시(학습 0회의 근거)",
        os.path.join(MITBIH, "mamba_data.npz"): "MIT-BIH 라벨·pid",
        os.path.join(MITBIH, "incart_data.npz"): "INCART 라벨·pid"}
miss = [p for p in need if not os.path.exists(p)]
if miss:
    raise AssetError(f"자산 없음: {miss}\n  → 실험22-A(CELL 4)를 먼저 돌려 캐시를 만들 것")
run.log("자산 확인 ✅ " + " · ".join(os.path.basename(p) for p in need))

P = dict(np.load(CACHE))
run.log(f"\n캐시 키: {sorted(P)}")
dm = np.load(os.path.join(MITBIH, "mamba_data.npz"))
di = np.load(os.path.join(MITBIH, "incart_data.npz"))
mpid, my = dm["pid"], dm["y"]
ipid, iy = di["pid"], di["y"]
TE = np.isin(mpid, _DS2)

# ★ 라벨을 캐시와 **대조**한다. 길이만 맞는 걸로는 안 된다(R10-b).
for nm, ycache, ylocal in (("within", P["y_within"], my[TE]), ("cross", P["y_cross"], iy)):
    a, b = np.asarray(ycache), np.asarray(ylocal)
    if a.shape != b.shape:
        raise AssetError(f"{nm} 라벨 길이 불일치 캐시 {a.shape} vs 로컬 {b.shape}")
    if not (a == b).all():
        raise AssetError(f"{nm} 라벨 **내용**이 다르다 — 캐시가 다른 실행의 것이다")
    h = np.bincount(a.astype(int), minlength=3)[:3]
    run.log(f"  {nm:<7} {len(a):>8,}비트 · N/S/V {h.tolist()} · 캐시=로컬 ✅")

COH = {"within (MIT-BIH DS2)": (np.asarray(P["y_within"]), np.asarray(mpid[TE])),
       "cross (INCART)":       (np.asarray(P["y_cross"]),  np.asarray(ipid))}
run.log(f"\n  레코드 수 — within {len(np.unique(mpid[TE]))} · cross {len(np.unique(ipid))}")
run.log("  ⚠️ INCART 의 '레코드' 75개는 실제 환자 32명이다(§6.5 L3). 아래 '환자' 는 레코드다")
run.save_json("config", CONFIG)

In [ ]:
# CELL 3 — 【Q1-A】 양성이 환자에게 어떻게 흩어져 있나
#   R11-3 의 '지배 지분' 을 코호트별로 실측하고, **누적 커버리지 곡선**을 본다.
#   소수 환자가 양성을 독점하면 전역 지표가 그 환자의 지표가 된다.
def per_record_stats(y, g, idx):
    """레코드별 (양성, 음성). 채점 가능 여부는 GMIN 이 정한다."""
    return {int(r): (int((y[g == r] == idx).sum()), int((g == r).sum() - (y[g == r] == idx).sum()))
            for r in np.unique(g)}

CLS = {"S": 1, "V": 2}
STATS = {}
run.log("\n" + "=" * 100)
run.log("【Q1-A】 양성의 환자 간 분포 — 전역 지표가 누구의 지표인가")
run.log("=" * 100)
for cn, (y, g) in COH.items():
    for c, idx in CLS.items():
        st = per_record_stats(y, g, idx)
        STATS[(cn, c)] = st
        pos = np.array(sorted((p for p, _ in st.values()), reverse=True))
        tot = pos.sum()
        nz = int((pos > 0).sum())
        cum = np.cumsum(pos) / max(tot, 1)
        k50 = int(np.searchsorted(cum, 0.50) + 1)
        k90 = int(np.searchsorted(cum, 0.90) + 1)
        run.log(f"\n  [{cn} · {c}]  양성 {tot:,} · 레코드 {len(st)}개(양성>0 인 곳 {nz}개)")
        run.log(f"    지배 지분 {pos[0]/max(tot,1):>6.1%}"
                f"  ·  상위 {k50}명이 50%  ·  상위 {k90}명이 90%")
        run.log(f"    상위 5명 양성 수 {pos[:5].tolist()} · 중앙값 {int(np.median(pos))}")
        run.log(f"    → 전역 지표 단독인용 "
                f"{'❌ 금지 (R11-3)' if pos[0]/max(tot,1) > DOM_MAX else '✅ 가능'}")

In [ ]:
# CELL 4 — 【Q1-B】 GMIN 스윕 — 커버리지와 해상도의 교환비를 실측한다
#
#   해상도를 두 가지로 잰다:
#     · R-prec **계단 크기** = 1/n_pos  — #213 사고(S 28개 → Δ +0.4857)의 직접 원인
#     · AUROC **표집 SE** (Hanley–McNeil 1982) — 지표에 덜 의존하는 기준
#   ⚠️ Hanley 는 보수적이다(부트 대비 1.1~1.6배). 하한 설정용으로만 쓰고 '진짜 SE' 로
#     인용하지 않는다. 아래 CELL 5 에서 실제 부트스트랩과 대조한다.
from sklearn.metrics import roc_auc_score

def hanley_se(auc, n_pos, n_neg):
    a = float(np.clip(auc, 1e-9, 1 - 1e-9))
    q1, q2 = a / (2 - a), 2 * a * a / (1 + a)
    v = a * (1 - a) + (n_pos - 1) * (q1 - a * a) + (n_neg - 1) * (q2 - a * a)
    return float(np.sqrt(max(v, 0.0) / (n_pos * n_neg)))

def sweep(y, g, idx, prob, gmins, st):
    tot = sum(p for p, _ in st.values())
    rows = []
    for gm in gmins:
        keep = [r for r, (p, n) in st.items() if p >= gm and n > 0]
        if not keep:
            rows.append(dict(gmin=gm, n_rec=0, cov=0.0, dom=np.nan,
                             step_med=np.nan, se_med=np.nan, se_max=np.nan)); continue
        pos = np.array([st[r][0] for r in keep])
        ses = []
        for r in keep:
            m = g == r
            t = (y[m] == idx)
            ses.append(hanley_se(roc_auc_score(t.astype(int), prob[m]), *st[r]))
        rows.append(dict(gmin=gm, n_rec=len(keep), cov=float(pos.sum() / max(tot, 1)),
                         dom=float(pos.max() / pos.sum()),
                         step_med=float(np.median(1.0 / pos)),
                         se_med=float(np.median(ses)), se_max=float(np.max(ses))))
    return rows

def pick(rows):
    """세 조건을 **전부** 만족하는 것 중 가장 낮은 GMIN(= 환자 최다)."""
    ok = [r for r in rows if r["n_rec"] >= N_MIN and r["se_med"] <= SE_CAP
          and r["se_max"] <= SE_CAP * SE_MULT]
    return min(ok, key=lambda r: r["gmin"]) if ok else None

# 대표 점수: v2·raw 의 시드 평균(리듬 포함 구성 — 실전에 쓸 구성)
SCORE = {"within (MIT-BIH DS2)": P["v2_within_raw"].mean(0),
         "cross (INCART)":       P["v2_cross_raw"].mean(0)}
run.log("\n" + "=" * 100)
run.log("【Q1-B】 GMIN 스윕 (점수 = v2·raw 시드평균)")
run.log("=" * 100)
SWEEP, PICK = {}, {}
for cn, (y, g) in COH.items():
    for c, idx in CLS.items():
        rows = sweep(y, g, idx, SCORE[cn][:, idx], GMINS, STATS[(cn, c)])
        SWEEP[(cn, c)] = rows
        sel = pick(rows); PICK[(cn, c)] = sel
        run.log(f"\n  [{cn} · {c}]")
        run.log(f"    {'GMIN':>5}{'환자':>6}{'커버리지':>9}{'지배지분':>9}"
                f"{'R-prec계단':>11}{'AUROC SE 중앙':>14}{'최대':>9}  조건")
        for r_ in rows:
            bad = []
            if r_["n_rec"] < N_MIN: bad.append(f"환자<{N_MIN}")
            if not (r_["se_med"] <= SE_CAP): bad.append("중앙SE")
            if not (r_["se_max"] <= SE_CAP * SE_MULT): bad.append("최대SE")
            if not (r_["cov"] >= COV_MIN): bad.append("커버리지")
            mark = "✅" if not bad else " · ".join(bad)
            star = " ★" if sel and r_["gmin"] == sel["gmin"] else ""
            run.log(f"    {r_['gmin']:>5}{r_['n_rec']:>6}{r_['cov']:>9.1%}{r_['dom']:>9.1%}"
                    f"{r_['step_med']:>11.4f}{r_['se_med']:>14.4f}{r_['se_max']:>9.4f}"
                    f"  {mark}{star}")
        run.log(f"    → 선택 {'GMIN=' + str(sel['gmin']) if sel else '**없음** — 조건을 만족하는 GMIN 이 없다'}")
        # ★ 교환비를 명시한다. GMIN 을 올리면 해상도는 얻지만 **지배 지분은 나빠진다**
        #   — 양성이 적은 레코드가 먼저 잘려나가기 때문이다. 둘은 같은 방향이 아니다.
        live = [r for r in rows if r["n_rec"] > 0]      # ★ 비었을 수 있다(양성 희박 코호트)
        if len(live) >= 2:
            d0, d1 = live[0]["dom"], live[-1]["dom"]
            run.log(f"      교환비: GMIN {live[0]['gmin']}→{live[-1]['gmin']} 에서"
                    f" 지배 지분 {d0:.1%} → {d1:.1%}"
                    f" {'(악화 — 해상도와 대표성이 반대로 간다)' if d1 > d0 else '(개선)'}")
        else:
            run.log("      교환비: 채점 가능한 GMIN 이 0~1개뿐 — 교환비를 낼 수 없다")

In [ ]:
# CELL 5 — 【Q1-C】 **부트스트랩 SE 로 다시 고른다** (Hanley 는 스크리닝으로 강등)
#
#  ★ 1차 실행이 드러낸 설계 결함: 이 셀이 CELL 4 의 **Hanley 기반 선택**에 의존했다.
#    Hanley 가 틀려 선택이 실패하면 **검증도 안 돌았다** — 순환 의존이다.
#    실제로 cross·S 는 "선택 없음" 이라 SE 를 재보지도 못한 채 기각됐다.
#
#  1차 실측: cross·V 의 Hanley/부트 중앙 비 = **2.76**. AUROC 가 천장(0.999~1.000)에
#    붙은 레코드에서 4~6배까지 벌어졌다 — 이항정규 가정이 깨지는 구간이다.
#    그 비를 cross·S 최대 SE 에 적용하면 0.1976 → 0.072 · 0.1341 → 0.049 로
#    **상한 0.10 을 통과한다.** 기각의 원인이 데이터가 아니라 근사식일 수 있다.
#
#  → 후보 GMIN 에서 **부트스트랩 SE 를 직접**(레코드 전수) 재고 그걸로 선택한다.
#    max 가 구속 조건이므로 표본이 아니라 **전수**여야 한다.
NB_BOOT  = 200
MAX_EVAL = 4       # 부트를 실제로 돌릴 최대 후보 수 — 전수 부트라 비용을 여기서 통제.
                   # ★ 처음엔 '가장 낮은 2개' 만 봤는데, 그 둘이 최대 SE 에서 걸리면
                   #   더 높은 GMIN 은 보지도 않고 '선택 없음' 이 됐다(픽스처에 걸림).
                   #   → **낮은 순으로 훑되 처음 통과하면 멈춘다**(조기 종료).
rng = np.random.RandomState(SEED0)

def boot_se(prob, y, g, recs, idx, nboot=NB_BOOT):
    """레코드별 AUROC 의 **부트스트랩 SE**. 비트를 복원추출한다(근사가 아니다)."""
    out = {}
    for r in recs:
        m = np.where(g == r)[0]
        t = (y[m] == idx); s = prob[m]
        vals = []
        for _ in range(nboot):
            j = rng.randint(0, len(m), len(m)); tj = t[j]
            if 0 < tj.sum() < len(tj):
                vals.append(roc_auc_score(tj.astype(int), s[j]))
        out[int(r)] = float(np.std(vals, ddof=1)) if len(vals) > 2 else np.nan
    return out

run.log("\n" + "=" * 100)
run.log(f"【Q1-C】 후보 GMIN 에서 **부트스트랩 SE**({NB_BOOT}회 · 레코드 전수)로 재선택")
run.log("=" * 100)
run.log("  Hanley 는 근사다. 1차에서 비가 2.76 이었으므로 **판정 근거에서 뺀다**.")
BOOT, PICK_B, RATIO = {}, {}, {}
for cn, (y, g) in COH.items():
    for c, idx in CLS.items():
        st, rows = STATS[(cn, c)], SWEEP[(cn, c)]
        live = [r for r in rows if r["n_rec"] > 0]
        cand = [r for r in live if r["n_rec"] >= N_MIN] or live      # 환자 조건 우선
        cand = sorted(cand, key=lambda r: r["gmin"])                 # 낮은 순 = 환자 많은 순
        run.log(f"\n  [{cn} · {c}] 후보 GMIN {[r['gmin'] for r in cand]}"
                f" · 환자 {[r['n_rec'] for r in cand]}  (최대 {MAX_EVAL}개까지 부트 검사)")
        best, n_eval = None, 0
        for r_ in cand:
            if best is not None or n_eval >= MAX_EVAL:
                break                                                # 조기 종료
            n_eval += 1
            keep = [k for k, (p, n) in st.items() if p >= r_["gmin"] and n > 0]
            bs = boot_se(SCORE[cn][:, idx], y, g, keep, idx)
            v = np.array([x for x in bs.values() if np.isfinite(x)])
            if not len(v):
                run.log(f"    GMIN {r_['gmin']:>3} · 부트 계산 불가(양성/음성 부족)"); continue
            bm, bx = float(np.median(v)), float(np.max(v))
            ratio = r_["se_med"] / max(bm, 1e-9)
            RATIO[(cn, c, r_["gmin"])] = ratio
            fails = []
            if r_["n_rec"] < N_MIN: fails.append(f"환자<{N_MIN}")
            if bm > SE_CAP: fails.append("중앙SE")
            if bx > SE_CAP * SE_MULT: fails.append("최대SE")
            if r_["cov"] < COV_MIN: fails.append("커버리지")
            if r_["dom"] > DOM_MAX: fails.append("지배")
            worst = max(bs, key=lambda k: (bs[k] if np.isfinite(bs[k]) else -1))
            run.log(f"    GMIN {r_['gmin']:>3} · 레코드 {len(keep):>3}"
                    f" | Hanley 중앙 {r_['se_med']:.4f} 최대 {r_['se_max']:.4f}"
                    f" | **부트 중앙 {bm:.4f} 최대 {bx:.4f}** | 비 {ratio:>4.2f}"
                    f"  {'✅ 전조건 통과' if not fails else '❌ ' + ' · '.join(fails)}")
            run.log(f"           최대 SE 는 #{worst}(양성 {st[worst][0]}개 · {bs[worst]:.4f})")
            if not fails and best is None:
                best = dict(r_, se_med_boot=bm, se_max_boot=bx, n_boot=NB_BOOT)
        PICK_B[(cn, c)] = best
        BOOT[(cn, c)] = {g: v for (a, b, g), v in RATIO.items() if (a, b) == (cn, c)}
        if best is None and n_eval >= MAX_EVAL and len(cand) > n_eval:
            run.log(f"    ⚠️ 후보 {len(cand)}개 중 {n_eval}개만 검사하고 멈췄다"
                    f" — 남은 GMIN {[r['gmin'] for r in cand[n_eval:]]} 는 미검사다."
                    " '없음' 이 아니라 **미확인**으로 읽을 것")
        run.log(f"    → 부트 기준 선택 {'GMIN=' + str(best['gmin']) if best else '**없음**'}")

_rs = [v for v in RATIO.values() if v is not None and np.isfinite(v)]
if _rs:
    run.log(f"\n  Hanley/부트 비 — 중앙 {np.median(_rs):.2f} · 범위 [{min(_rs):.2f}, {max(_rs):.2f}]")
    run.log("  → 1 보다 크면 Hanley 가 보수적이다. 2 를 넘으면 **환자를 불필요하게 잃는다**.")
    run.log("     그래서 판정은 부트로 하고 Hanley 는 스윕의 빠른 스크리닝에만 쓴다.")

In [ ]:
# CELL 6 — 【Q1 채점】 사전등록 관문 (판정 근거 = **부트스트랩** SE · CELL 5)
run.log("\n" + "=" * 100)
run.log("【Q1 사전등록 채점】  ※ 판정은 부트스트랩 SE 로 한다(Hanley 는 참고)")
run.log("=" * 100)
TARGET, CTRL = ("cross (INCART)", "S"), ("within (MIT-BIH DS2)", "S")
sel = PICK_B[TARGET]
VERD = {}

def g_(name, ok, detail):
    VERD[name] = "✅ 지지" if ok else "❌ 기각"
    run.log(f"  {name:<7}{VERD[name]}  {detail}")

if sel is None:
    for k in ("Q1-1", "Q1-2", "Q1-3", "Q1-4"):
        VERD[k] = "❌ 기각"
    liv = [r for r in SWEEP[TARGET] if r["n_rec"] > 0]
    top = max(liv, key=lambda r: r["n_rec"]) if liv else None
    run.log(f"  ❌ INCART S 에서 조건을 **동시에** 만족하는 GMIN 이 없다.")
    if top:
        run.log(f"     가장 환자가 많은 지점: GMIN={top['gmin']} → {top['n_rec']}명 "
                f"(커버 {top['cov']:.1%} · 지배 {top['dom']:.1%})")
    run.log("     → 이 코호트로도 S 판정은 불가능하다. Q7(SVDB)·실험21(Chapman) 로 간다.")
else:
    g_("Q1-1", sel["n_rec"] >= N_MIN,
       f"INCART S 채점 환자 **{sel['n_rec']}명** (GMIN={sel['gmin']}) ≥ {N_MIN}"
       f"  · 실험22-A 는 11명이었다")
    g_("Q1-2", sel["se_med_boot"] <= SE_CAP,
       f"환자별 AUROC **부트** 중앙 SE **{sel['se_med_boot']:.4f}** ≤ {SE_CAP}"
       f"  (최대 {sel['se_max_boot']:.4f} ≤ {SE_CAP*SE_MULT})")
    g_("Q1-3", sel["cov"] >= COV_MIN,
       f"채점 환자가 덮는 S **{sel['cov']:.1%}** ≥ {COV_MIN:.0%}")
    g_("Q1-4", sel["dom"] <= DOM_MAX,
       f"채점 환자 안 지배 지분 **{sel['dom']:.1%}** ≤ {DOM_MAX:.0%}")

# ── Q1-5 는 **같은 GMIN 에서** 비교한다.
#   1차에서 'INCART 0명 vs DS2 0명' 으로 기각했는데, 그건 '선택이 없다' 를 '환자가 없다' 로
#   잘못 읽은 것이다. 코호트 비교는 선택 성공 여부와 무관하게 성립한다.
run.log("\n  ── Q1-5 코호트 비교 (같은 GMIN 에서) ──")
wx = {r["gmin"]: r["n_rec"] for r in SWEEP[CTRL]}
xx = {r["gmin"]: r["n_rec"] for r in SWEEP[TARGET]}
common = [g for g in GMINS if g in wx and g in xx and (wx[g] or xx[g])]
run.log(f"    {'GMIN':>5}{'INCART':>8}{'DS2':>7}   판정")
for g in common:
    run.log(f"    {g:>5}{xx[g]:>8}{wx[g]:>7}   {'INCART 우세' if xx[g] > wx[g] else '동률/열세'}")
win = sum(1 for g in common if xx[g] > wx[g])
g_("Q1-5", win == len(common),
   f"**{win}/{len(common)}** 개 GMIN 에서 INCART 가 많다"
   f"  (DS2 최대 {max(wx.values())}명 · INCART 최대 {max(xx.values())}명)")

run.log("\n  " + "  ".join(f"{k}: {v}" for k, v in VERD.items()))
run.log(f"\n  ⚠️ 여기서 '환자' 는 INCART 의 **레코드**다(실제 환자 32명 · §6.5 L3).")
run.log("  ⚠️ DS2 는 S>0 인 레코드가 16/22 이고 중앙값이 4개다 →"
        f" **어떤 GMIN 을 써도 {N_MIN}명은 불가능**하다(구조적 상한 {max(wx.values())}명).")
CONFIG["result"] = {
    "verdicts": VERD,
    "picked_boot": {f"{a} · {b}": (None if v is None else
                    {k: v[k] for k in ("gmin", "n_rec", "cov", "dom",
                                       "se_med_boot", "se_max_boot", "n_boot")})
                    for (a, b), v in PICK_B.items()},
    "picked_hanley": {f"{a} · {b}": (None if v is None else
                      {k: v[k] for k in ("gmin", "n_rec", "se_med", "se_max")})
                      for (a, b), v in PICK.items()},
    "hanley_over_boot": {f"{a} · {b} · GMIN{c}": (None if v is None else round(v, 3))
                         for (a, b, c), v in RATIO.items()},
    "cohort_compare": {int(g): {"incart": xx[g], "ds2": wx[g]} for g in common},
    "sweep": {f"{a} · {b}": rows for (a, b), rows in SWEEP.items()}}
run.save_json("config", CONFIG)

In [ ]:
# CELL 7 — 그림 + 저장
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
for (cn, c), rows in SWEEP.items():
    if c != "S":
        continue
    x = [r["gmin"] for r in rows]
    ax[0].plot(x, [r["n_rec"] for r in rows], "o-", label=cn)
    ax[1].plot(x, [r["cov"] for r in rows], "o-", label=cn)
    ax[2].plot(x, [r["se_med"] for r in rows], "o-", label=cn)
ax[0].axhline(N_MIN, ls="--", c="crimson"); ax[0].set_ylabel("채점 환자 수")
ax[1].axhline(COV_MIN, ls="--", c="crimson"); ax[1].set_ylabel("S 커버리지")
ax[2].axhline(SE_CAP, ls="--", c="crimson"); ax[2].set_ylabel("환자별 AUROC 중앙 SE")
for a, t in zip(ax, ("① 환자 수 (많을수록 좋다)", "② 커버리지 (높을수록 좋다)",
                     "③ 해상도 (낮을수록 좋다)")):
    a.set_xlabel("GMIN_S (환자별 최소 양성 수)"); a.set_xscale("log")
    a.set_title(t); a.grid(alpha=.3); a.legend(fontsize=8)
fig.suptitle("Q1 — GMIN 을 올리면 ①은 잃고 ③은 얻는다. 교환비를 실측한다", y=1.02)
fig.tight_layout()
run.save_fig("q1_gmin_tradeoff", fig)
plt.show()
run.log("\n저장 완료 — " + run.dir)
run.log("다음: `ingest_run.py --quest ailab-2026-0046 --step gmin-resolution` 으로 로그를 쌓는다")